In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
import pytest
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date

In [0]:
# Tests for get_value_from_dict

def test_get_value_from_dict_top_level():
    d = {"a": 1, "b": 2}
    assert get_value_from_dict(d, "a") == 1

def test_get_value_from_dict_nested():
    d = {"level1": {"level2": {"target": "found"}}}
    assert get_value_from_dict(d, "target") == "found"

def test_get_value_from_dict_missing_key():
    d = {"a": 1}
    assert get_value_from_dict(d, "z") is None

def test_get_value_from_dict_invalid_input():
    with pytest.raises(TypeError):
        get_value_from_dict("not_a_dict", "key")

In [0]:
# Tests for fill_defaults

def test_fill_defaults_fills_nulls():
    df = spark.createDataFrame([(1, None), (2, "b")], ["id", "name"])
    result = fill_defaults(df, {"name": "unknown"})
    rows = result.collect()
    assert rows[0]["name"] == "unknown"
    assert rows[1]["name"] == "b"

def test_fill_defaults_empty_defaults_returns_same():
    df = spark.createDataFrame([(1, None)], ["id", "name"])
    result = fill_defaults(df, {})
    assert result.collect() == df.collect()

def test_fill_defaults_none_df_raises():
    with pytest.raises(ValueError):
        fill_defaults(None, {"name": "x"})

In [0]:
# Tests for remove_nulls_for_key

def test_remove_nulls_single_key():
    df = spark.createDataFrame([(1, "a"), (None, "b"), (3, "c")], ["id", "val"])
    result = remove_nulls_for_key(df, "id")
    assert result.count() == 2

def test_remove_nulls_multiple_keys():
    df = spark.createDataFrame([(1, "a"), (None, "b"), (3, None)], ["id", "code"])
    result = remove_nulls_for_key(df, ["id", "code"])
    assert result.count() == 1

def test_remove_nulls_invalid_key_type():
    df = spark.createDataFrame([(1,)], ["id"])
    with pytest.raises(TypeError):
        remove_nulls_for_key(df, 123)

In [0]:
# Tests for parse_dates

def test_parse_dates_valid_format():
    df = spark.createDataFrame([("15/3/2024",)], ["order_date"])
    result = parse_dates(df, {"order_date": "d/M/yyyy"})
    row = result.collect()[0]
    assert row["order_date"] == date(2024, 3, 15)

def test_parse_dates_null_passthrough():
    df = spark.createDataFrame([(None,)], schema=StructType([StructField("d", StringType())]))
    result = parse_dates(df, {"d": "yyyy-MM-dd"})
    assert result.collect()[0]["d"] is None

In [0]:
# Tests for apply_schema

def test_apply_schema_selects_columns():
    df = spark.createDataFrame([(1, "a", 3.0)], ["id", "name", "extra"])
    schema = StructType([StructField("id", IntegerType()), StructField("name", StringType())])
    result = apply_schema(df, schema)
    assert result.columns == ["id", "name"]

def test_apply_schema_none_df_raises():
    schema = StructType([StructField("id", IntegerType())])
    with pytest.raises(ValueError):
        apply_schema(None, schema)

In [0]:
# Tests for apply_schema_alias

def test_apply_schema_alias_renames():
    df = spark.createDataFrame([(1, "a")], ["old_id", "old_name"])
    result = apply_schema_alias(df, {"old_id": "new_id", "old_name": "new_name"})
    assert result.columns == ["new_id", "new_name"]

def test_apply_schema_alias_partial():
    df = spark.createDataFrame([(1, "a")], ["id", "name"])
    result = apply_schema_alias(df, {"name": "full_name"})
    assert result.columns == ["id", "full_name"]

In [0]:
def test_utils_main():
    # Run all tests

    test_functions = [
        test_get_value_from_dict_top_level,
        test_get_value_from_dict_nested,
        test_get_value_from_dict_missing_key,
        test_get_value_from_dict_invalid_input,
        test_fill_defaults_fills_nulls,
        test_fill_defaults_empty_defaults_returns_same,
        test_fill_defaults_none_df_raises,
        test_remove_nulls_single_key,
        test_remove_nulls_multiple_keys,
        test_remove_nulls_invalid_key_type,
        test_parse_dates_valid_format,
        test_parse_dates_null_passthrough,
        test_apply_schema_selects_columns,
        test_apply_schema_none_df_raises,
        test_apply_schema_alias_renames,
        test_apply_schema_alias_partial,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")
